# Bag of Words com DataLoader

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


O código abaixo implementa um classificador de texto bag of words simples.

- Tokenizamos o texto, criamos um vocabulário e codificamos cada trecho do conjunto
- Um lookup extrai embeddings para cada token de entrada
- Os vetores de embedding são somados a um vetor de bias
- O vetor resultante são os scores
- Aplica-se softmax aos scores para gerar probabilidades de classificação

A diferença em relação ao `bow.ipynb` é o uso do `DataLoader` do PyTorch.

![img txt](../img/bow.png?raw=true)


In [ ]:
import torch
import random
import torch.nn as nn

### Baixar Dados


In [ ]:
%%capture

# download the files

# create the data folders
!mkdir data data/classes
!cp dev.txt data/classes
!cp test.txt data/classes
!cp train.txt data/classes

### Ler Dados


In [ ]:
# function to read in data, process each line and split columns by " ||| "
def read_data(filename):
    data = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.lower().strip()
            line = line.split(' ||| ')
            data.append(line)
    return data

train_data = read_data('data/classes/train.txt')
test_data = read_data('data/classes/test.txt')

### Construir Vocabulário e Datasets


In [ ]:
# creating the word and tag indices
word_to_index = {}
word_to_index["<unk>"] = len(word_to_index) # adds <UNK> to dictionary
tag_to_index = {}

# create word to index dictionary and tag to index dictionary from data
def create_dict(data, check_unk=False):
    for line in data:
        for word in line[1].split(" "):
            if check_unk == False:
                if word not in word_to_index:
                    word_to_index[word] = len(word_to_index)
            else:
                if word not in word_to_index:
                    word_to_index[word] = word_to_index["<unk>"]

        if line[0] not in tag_to_index:
            tag_to_index[line[0]] = len(tag_to_index)

create_dict(train_data)
create_dict(test_data, check_unk=True)

# create word and tag tensors from data
def create_tensor(data):
    for line in data:
        yield [[word_to_index[word] for word in line[1].split(" ")], tag_to_index[line[0]]]

train_data = [*create_tensor(train_data)]
test_data = [*create_tensor(test_data)]

number_of_words = len(word_to_index)
number_of_tags = len(tag_to_index)

### Converter Dados para `Dataset` do PyTorch


In [ ]:
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

# load data into a dataset and dataloader; ensure that the data is split X, y
class TextDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.as_tensor(self.data[idx][0]), torch.as_tensor(self.data[idx][1])

train_dataset = TextDataset(train_data)
test_dataset = TextDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)


### Modelo


In [ ]:
# cpu or gpu
device = "cuda" if torch.cuda.is_available() else "cpu"

# create a simple neural network with embedding layer, bias, and xavier initialization
class BoW(torch.nn.Module):
    def __init__(self, nwords, ntags):
        super(BoW, self).__init__()
        self.embedding = nn.Embedding(nwords, ntags)
        nn.init.xavier_uniform_(self.embedding.weight)

        type = torch.cuda.FloatTensor if torch.cuda.is_available() else torch.FloatTensor
        self.bias = torch.zeros(ntags, requires_grad=True).type(type)

    def forward(self, x):
        emb = self.embedding(x) # seq_len x ntags (for each seq) 
        out = torch.sum(emb, dim=0) + self.bias # ntags
        out = out.view(1, -1) # reshape to (1, ntags)
        return out

### Treinar o Modelo


In [ ]:
# train and test the BoW model
model = BoW(number_of_words, number_of_tags).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())
type = torch.LongTensor

if torch.cuda.is_available():
    model.to(device)
    type = torch.cuda.LongTensor

# perform training of the Bow model
def train_bow(model, optimizer, criterion, train_data):
    for ITER in range(10):
        # perform training
        model.train()
        total_loss = 0.0
        train_correct = 0
        for batch, (sentence, tag) in enumerate(train_loader):
            sentence = sentence[0].to(device)
            tag = tag.to(device)

            output = model(sentence)
            predicted = torch.argmax(output.data.detach()).item()
            
            loss = criterion(output, tag)
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if predicted == tag: train_correct+=1

        # perform testing of the model
        model.eval()
        test_correct = 0
        for batch, (sentence, tag) in enumerate(test_loader):
            sentence = sentence[0].to(device)
            output = model(sentence)
            predicted = torch.argmax(output.data.detach()).item()
            if predicted == tag: test_correct += 1
        
        # print model performance results
        log = f'ITER: {ITER+1} | ' \
            f'train loss/sent: {total_loss/len(train_data):.4f} | ' \
            f'train accuracy: {train_correct/len(train_data):.4f} | ' \
            f'test accuracy: {test_correct/len(test_data):.4f}'
        print(log)

# call the train_bow function
train_bow(model, optimizer, criterion, train_data)

### Exercícios

Para continuar praticando:

- Use diferentes tamanhos de batch e veja como afeta o treinamento.
- Use [`torchtext`](https://pytorch.org/text/stable/index.html#) para carregar outros conjuntos e criar tokenizer + vocabulário.
- Escreva uma mini-biblioteca em Python para treinar e avaliar modelos. O código deste notebook serve como ponto de partida.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
